In [1]:
import numpy as np
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

train = np.loadtxt("Datasets/pendigits.tra", delimiter=",")
test = np.loadtxt("Datasets/pendigits.tes", delimiter=",")
Xtr, ytr = train[:, :16] / 100, train[:, 16].astype(int)
Xte, yte = test[:, :16] / 100, test[:, 16].astype(int)

tree = DecisionTreeClassifier(max_depth=12, random_state=42)
knn = KNeighborsClassifier(n_neighbors=5)

stack = StackingClassifier(estimators=[("tree", tree), ("knn", knn)], final_estimator=LogisticRegression(max_iter=500), cv=5)
stack.fit(Xtr, ytr)
acc_stack = accuracy_score(yte, stack.predict(Xte))

Xa, Xb, ya, yb = train_test_split(Xtr, ytr, test_size=0.2, stratify=ytr, random_state=42)
tree.fit(Xa, ya)
knn.fit(Xa, ya)

print("Decision Tree Accuracy:", round(accuracy_score(yte, tree.predict(Xte)), 4))
print("k-NN Accuracy:", round(accuracy_score(yte, knn.predict(Xte)), 4))

meta_train = np.column_stack([tree.predict_proba(Xb), knn.predict_proba(Xb)])
meta_test = np.column_stack([tree.predict_proba(Xte), knn.predict_proba(Xte)])

meta = LogisticRegression(max_iter=500)
meta.fit(meta_train, yb)
acc_blend = accuracy_score(yte, meta.predict(meta_test))

print("Stacking Accuracy:", round(acc_stack, 4))
print("Blending Accuracy:", round(acc_blend, 4))


C:\Users\Shubham\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\Shubham\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Shubham\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Shubham\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close

Decision Tree Accuracy: 0.9125
k-NN Accuracy: 0.9768
Stacking Accuracy: 0.978
Blending Accuracy: 0.9754
